# Detector export to ONNX with a parity check against PyTorch

## 0. Settings, packages, mount Drive

In [1]:
!pip -q install ultralytics onnx onnxslim onnxruntime-gpu

In [2]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'   # the only Drive path
# final weights: P09_Model_Freeze.json -> primary_model Joint_StrongerAug, input 640
DET_WEIGHTS = '03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt'
MODEL_NAME  = 'YOLO11n_JointStrongerAug'
IMGSZ, OPSET = 640, 17
FRAMES_DIR  = '04_Segmentation/seg/yolo_dataset/images/val'   # real frames for the parity check (jpg/png); None = random only
N_FRAMES, N_RANDOM, SEED = 50, 16, 42
PARITY_ATOL = 1e-3                     # acceptance: max abs diff <= 1e-3
CONF, NMS_IOU = 0.25, 0.7              # for the end-to-end detection comparison
LAT_WARMUP, LAT_RUNS = 20, 200

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports, paths, output folder

In [3]:
import os, re, sys, json, time, shutil, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch, cv2
import ultralytics, onnx, onnxruntime as ort
from ultralytics import YOLO

ROOT = Path(DRIVE_PROJECT)
P = lambda x: Path(x) if str(x).startswith('/') else ROOT / x
def rel(p):
    try: return Path(p).relative_to(ROOT).as_posix()
    except ValueError: return str(p)
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
W = P(DET_WEIGHTS)
if not W.is_file(): raise FileNotFoundError(f'detector weights not found: {W}')

TODAY = time.strftime('%Y-%m-%d')
EXP = ROOT / '03_Detection/Export'; EXP.mkdir(parents=True, exist_ok=True)
VERSION = max([int(m.group(1)) for f in os.listdir(EXP) if (m := re.fullmatch(rf'{TODAY}_{MODEL_NAME}_v(\d+)', f))], default=0) + 1
OUT = EXP / f'{TODAY}_{MODEL_NAME}_v{VERSION}'; OUT.mkdir()
WORK = Path('/content/det_export'); shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir()
shutil.copy2(W, WORK / 'best.pt')            # export happens on a local copy: nothing is written next to the original weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False   # plain fp32 reference
VERSIONS = {'python': sys.version.split()[0], 'torch': torch.__version__, 'ultralytics': ultralytics.__version__,
            'onnx': onnx.__version__, 'onnxruntime': ort.__version__, 'providers': ort.get_available_providers(), 'gpu': GPU}
print(json.dumps(VERSIONS, indent=1)); print('weights:', rel(W), '\noutput:', rel(OUT))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
{
 "python": "3.13.16",
 "torch": "2.14.1+cu130",
 "ultralytics": "8.4.215",
 "onnx": "1.23.2",
 "onnxruntime": "1.32.0",
 "providers": [
  "TensorrtExecutionProvider",
  "CUDAExecutionProvider",
  "CPUExecutionProvider"
 ],
 "gpu": "Tesla T4"
}
weights: 03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt 
output: 03_Detection/Export/2026-11-16_YOLO11n_JointStrongerAug_v1


## 2. Export to ONNX

In [4]:
yolo = YOLO(str(WORK / 'best.pt'))
NAMES = [yolo.names[i] for i in sorted(yolo.names)]
onnx_tmp = yolo.export(format='onnx', imgsz=IMGSZ, opset=OPSET, dynamic=False, simplify=True, half=False, nms=False, device='cpu')
ONNX = OUT / 'model.onnx'; shutil.copy2(onnx_tmp, ONNX)
m = onnx.load(str(ONNX)); onnx.checker.check_model(m)
dims = lambda t: [d.dim_param or d.dim_value for d in t.type.tensor_type.shape.dim]
IO = {'inputs': [(t.name, dims(t)) for t in m.graph.input], 'outputs': [(t.name, dims(t)) for t in m.graph.output],
      'opset': max(o.version for o in m.opset_import if o.domain in ('', 'ai.onnx'))}
del m
print('classes:', NAMES); print(IO); print(f'model.onnx {ONNX.stat().st_size / 1e6:.1f} MB')

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This model has no one-to-one head; using one-to-many outputs.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,598,752 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/det_export/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (5.2 MB)

ONNX: starting export with onnx 1.23.2 opset 17...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 10.1s, saved as '/content/det_export/best.onnx' (10.2 MB)

Export complete (12.0s)
Results saved to /content/det_export/best.onnx
Predict:         yolo predict task=detect model=/content/det_export/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/det

## 3. Parity inputs (letterboxed frames + random tensors)

In [5]:
def letterbox(img_bgr, size=IMGSZ, color=114):
    h, w = img_bgr.shape[:2]; r = min(size / h, size / w)
    nh, nw = round(h * r), round(w * r)
    im = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, left = (size - nh) // 2, (size - nw) // 2
    out = np.full((size, size, 3), color, np.uint8); out[top:top + nh, left:left + nw] = im
    return out
def to_input(img_bgr):   # same as Ultralytics: BGR -> RGB, HWC -> CHW, /255, float32
    return np.ascontiguousarray(letterbox(img_bgr)[..., ::-1].transpose(2, 0, 1), dtype=np.float32)[None] / 255.0

rng = np.random.default_rng(SEED)
INPUTS, SRC, FRAME_PATHS = [], [], []
if FRAMES_DIR and P(FRAMES_DIR).is_dir():
    allf = sorted(p for p in P(FRAMES_DIR).rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    FRAME_PATHS = [allf[i] for i in sorted(rng.choice(len(allf), min(N_FRAMES, len(allf)), replace=False))] if allf else []
    for f in FRAME_PATHS:
        im = cv2.imread(str(f))
        if im is not None: INPUTS.append(to_input(im)); SRC.append(f.name)
for i in range(N_RANDOM):
    INPUTS.append(rng.random((1, 3, IMGSZ, IMGSZ), dtype=np.float32)); SRC.append(f'random_{i}')
print(f'{len(INPUTS)} inputs: {len(FRAME_PATHS)} frames from {FRAMES_DIR}, {N_RANDOM} random')

46 inputs: 30 frames from 04_Segmentation/seg/yolo_dataset/images/val, 16 random


## 4. Raw-output parity: PyTorch fp32 vs ONNX Runtime

In [6]:
torch_model = YOLO(str(WORK / 'best.pt')).model.float().eval()
def torch_out(x, dev):
    with torch.no_grad():
        y = torch_model.to(dev)(torch.from_numpy(x).to(dev))
    return (y[0] if isinstance(y, (list, tuple)) else y).float().cpu().numpy()

NC = len(NAMES); PARITY, ROWS = {}, []
for prov, dev in (('CPUExecutionProvider', 'cpu'), ('CUDAExecutionProvider', 'cuda')):
    if prov not in ort.get_available_providers() or (dev == 'cuda' and DEVICE != 'cuda'):
        PARITY[prov] = {'available': False}; print(prov, 'not available'); continue
    sess = ort.InferenceSession(str(ONNX), providers=[prov])
    if sess.get_providers()[0] != prov:
        PARITY[prov] = {'available': False, 'reason': f'fell back to {sess.get_providers()}'}; print(prov, 'fell back'); continue
    iname = sess.get_inputs()[0].name
    dmax = dbox = dcls = 0.0; dsum = n = 0
    for x, s in zip(INPUTS, SRC):
        a = torch_out(x, dev); b = sess.run(None, {iname: x})[0]
        assert a.shape == b.shape, (a.shape, b.shape)
        d = np.abs(a.astype(np.float64) - b.astype(np.float64))
        dmax = max(dmax, float(d.max())); dbox = max(dbox, float(d[:, :4].max())); dcls = max(dcls, float(d[:, 4:].max()))
        dsum += float(d.sum()); n += d.size
        ROWS.append({'provider': prov, 'input': s, 'max_abs_diff': float(d.max()), 'box_max': float(d[:, :4].max()),
                     'class_score_max': float(d[:, 4:].max()), 'mean_abs_diff': float(d.mean())})
    PARITY[prov] = {'available': True, 'output_shape': list(a.shape), 'max_abs_diff': dmax, 'box_coords_max_abs_diff_px': dbox,
                    'class_scores_max_abs_diff': dcls, 'mean_abs_diff': dsum / n, 'inputs': len(INPUTS),
                    'pass': dmax <= PARITY_ATOL}
    print(f"{prov}: max abs diff {dmax:.3e} (boxes {dbox:.3e} px, scores {dcls:.3e}) -> {'PASS' if dmax <= PARITY_ATOL else 'FAIL'} (<= {PARITY_ATOL})")
torch_model.to('cpu')
pd.DataFrame(ROWS).to_csv(OUT / 'parity_per_input.csv', index=False)

CPUExecutionProvider: max abs diff 4.074e-03 (boxes 4.074e-03 px, scores 3.517e-06) -> FAIL (<= 0.001)
CUDAExecutionProvider: max abs diff 3.754e-03 (boxes 3.754e-03 px, scores 4.023e-06) -> FAIL (<= 0.001)


## 5. End-to-end detections: PyTorch vs ONNX (after NMS)

In [7]:
def iou(a, b):
    x1, y1 = np.maximum(a[:, None, 0], b[None, :, 0]), np.maximum(a[:, None, 1], b[None, :, 1])
    x2, y2 = np.minimum(a[:, None, 2], b[None, :, 2]), np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)
m_pt, m_ox = YOLO(str(WORK / 'best.pt')), YOLO(str(ONNX), task='detect')
E2E = []
for f in FRAME_PATHS:
    kw = dict(imgsz=IMGSZ, conf=CONF, iou=NMS_IOU, verbose=False, device=0 if DEVICE == 'cuda' else 'cpu')
    a, b = m_pt.predict(str(f), **kw)[0].boxes, m_ox.predict(str(f), **kw)[0].boxes
    A, B = a.xyxy.cpu().numpy(), b.xyxy.cpu().numpy()
    ca, cb = a.cls.cpu().numpy().astype(int), b.cls.cpu().numpy().astype(int)
    matched = same_cls = 0; conf_d = []
    if len(A) and len(B):
        M = iou(A, B); used = set()
        for i in np.argsort(-a.conf.cpu().numpy()):
            j = int(M[i].argmax())
            if M[i, j] >= 0.9 and j not in used:
                used.add(j); matched += 1; same_cls += ca[i] == cb[j]
                conf_d.append(abs(float(a.conf[i]) - float(b.conf[j])))
    E2E.append({'frame': f.name, 'pytorch_boxes': len(A), 'onnx_boxes': len(B), 'matched_iou_0.9': matched,
                'same_class': int(same_cls), 'max_conf_diff': max(conf_d) if conf_d else 0.0})
E2E = pd.DataFrame(E2E)
if len(E2E):
    tot = E2E[['pytorch_boxes', 'onnx_boxes', 'matched_iou_0.9', 'same_class']].sum().to_dict()
    E2E_SUM = {**{k: int(v) for k, v in tot.items()}, 'max_conf_diff': float(E2E.max_conf_diff.max()), 'frames': len(E2E)}
    E2E.to_csv(OUT / 'detections_pytorch_vs_onnx.csv', index=False)
else:
    E2E_SUM = {'frames': 0, 'note': 'no frames given'}
print(E2E_SUM)

Loading /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/03_Detection/Export/2026-11-16_YOLO11n_JointStrongerAug_v1/model.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.32.0 with CUDAExecutionProvider
{'pytorch_boxes': 987, 'onnx_boxes': 977, 'matched_iou_0.9': 876, 'same_class': 876, 'max_conf_diff': 0.2070463001728058, 'frames': 30}


## 5b. Same tensor into both backends, then the same NMS

In [8]:
# Same 640x640 tensor into PyTorch and ONNX Runtime, then the SAME NMS on both raw outputs.
# predict() on a file lets each backend letterbox on its own (PyTorch: rect, e.g. 640x384; static ONNX: 640x640),
# so section 5 compares different inputs. Here only the backend differs.
try:
    from ultralytics.utils.nms import non_max_suppression
except ImportError:
    from ultralytics.utils.ops import non_max_suppression

def nms(raw):   # raw [1, 4+nc, N] -> [n, 6] x1 y1 x2 y2 conf cls
    return non_max_suppression(torch.from_numpy(raw).float(), conf_thres=CONF, iou_thres=NMS_IOU, max_det=300)[0].numpy()

FRAME_INPUTS = [(x, s) for x, s in zip(INPUTS, SRC) if not s.startswith('random_')]
SAME_T, SAME_ROWS = {}, []
for prov, dev in (('CUDAExecutionProvider', 'cuda'), ('CPUExecutionProvider', 'cpu')):
    if not PARITY.get(prov, {}).get('available'): continue
    sess = ort.InferenceSession(str(ONNX), providers=[prov]); iname = sess.get_inputs()[0].name
    tot = dict(pytorch_boxes=0, onnx_boxes=0, matched_iou_0_9=0, same_class=0); cdiff = bdiff = 0.0
    for x, s in FRAME_INPUTS:
        A, B = nms(torch_out(x, dev)), nms(sess.run(None, {iname: x})[0])
        m = 0; same = 0
        if len(A) and len(B):
            M = iou(A[:, :4], B[:, :4]); used = set()
            for i in np.argsort(-A[:, 4]):
                j = int(M[i].argmax())
                if M[i, j] >= 0.9 and j not in used:
                    used.add(j); m += 1; same += int(A[i, 5] == B[j, 5])
                    cdiff = max(cdiff, abs(float(A[i, 4] - B[j, 4]))); bdiff = max(bdiff, float(np.abs(A[i, :4] - B[j, :4]).max()))
        tot['pytorch_boxes'] += len(A); tot['onnx_boxes'] += len(B); tot['matched_iou_0_9'] += m; tot['same_class'] += same
        SAME_ROWS.append({'provider': prov, 'frame': s, 'pytorch_boxes': len(A), 'onnx_boxes': len(B), 'matched_iou_0_9': m, 'same_class': same})
    SAME_T[prov] = {**tot, 'frames': len(FRAME_INPUTS), 'max_conf_diff': cdiff, 'max_box_diff_px': bdiff, 'conf': CONF, 'nms_iou': NMS_IOU}
    print(f"{prov}: {SAME_T[prov]}")
torch_model.to('cpu')
pd.DataFrame(SAME_ROWS).to_csv(OUT / 'detections_same_tensor.csv', index=False)

CUDAExecutionProvider: {'pytorch_boxes': 977, 'onnx_boxes': 977, 'matched_iou_0_9': 977, 'same_class': 977, 'frames': 30, 'max_conf_diff': 4.023313522338867e-06, 'max_box_diff_px': 0.0001220703125, 'conf': 0.25, 'nms_iou': 0.7}
CPUExecutionProvider: {'pytorch_boxes': 977, 'onnx_boxes': 977, 'matched_iou_0_9': 977, 'same_class': 977, 'frames': 30, 'max_conf_diff': 3.516674041748047e-06, 'max_box_diff_px': 0.0001220703125, 'conf': 0.25, 'nms_iou': 0.7}


## 6. Latency (batch 1, 640)

In [9]:
def timeit(fn):
    for _ in range(LAT_WARMUP): fn()
    t = []
    for _ in range(LAT_RUNS):
        t0 = time.perf_counter(); fn(); t.append((time.perf_counter() - t0) * 1e3)
    return {'p50_ms': round(float(np.percentile(t, 50)), 2), 'p95_ms': round(float(np.percentile(t, 95)), 2)}
x = INPUTS[0]; LAT = {}
for prov in ('CUDAExecutionProvider', 'CPUExecutionProvider'):
    if PARITY.get(prov, {}).get('available'):
        s = ort.InferenceSession(str(ONNX), providers=[prov]); n = s.get_inputs()[0].name
        LAT[f'onnx {prov[:-17]}'] = timeit(lambda: s.run(None, {n: x}))
if DEVICE == 'cuda':
    tm = torch_model.to('cuda'); xt = torch.from_numpy(x).cuda()
    def f():
        with torch.no_grad(): tm(xt)
        torch.cuda.synchronize()
    LAT['pytorch CUDA fp32'] = timeit(f); torch_model.to('cpu')
print(pd.DataFrame(LAT).T.to_string())

                   p50_ms  p95_ms
onnx CUDA            5.37    6.34
onnx CPU            98.80  147.07
pytorch CUDA fp32   11.51   20.32


## 7. Parity report

In [10]:
avail = {k: v for k, v in PARITY.items() if v.get('available')}
MAXD = max(v['max_abs_diff'] for v in avail.values())
ACCEPT = {'criterion': f'ONNX parity max abs diff <= {PARITY_ATOL}', 'max_abs_diff': MAXD, 'met': MAXD <= PARITY_ATOL,
          'per_provider': {k: v['pass'] for k, v in avail.items()}}
REPORT = {'created': time.strftime('%Y-%m-%d %H:%M'), 'output': rel(OUT),
          'weights': {'path': rel(W), 'sha256': sha256(W), 'selection': 'P09_Model_Freeze.json: primary_model Joint_StrongerAug'},
          'onnx': {'file': 'model.onnx', 'sha256': sha256(ONNX), 'imgsz': IMGSZ, 'opset': IO['opset'], 'io': IO,
                   'classes': NAMES, 'nms_in_graph': False, 'dynamic': False, 'half': False},
          'preprocess': 'letterbox to 640 (pad 114), BGR->RGB, CHW, /255, float32',
          'output_layout': f'[1, 4 + {len(NAMES)}, N]: cx, cy, w, h (pixels of the 640 input), then class scores',
          'parity_inputs': {'frames': len(FRAME_PATHS), 'frames_dir': FRAMES_DIR, 'random': N_RANDOM, 'seed': SEED},
          'parity': PARITY, 'acceptance': ACCEPT,
          'detections_after_nms_predict_own_letterbox': E2E_SUM,
          'detections_after_nms_same_tensor': SAME_T, 'latency_ms': LAT, 'versions': VERSIONS}
(OUT / 'parity_report.json').write_text(json.dumps(REPORT, indent=1, default=str))
rows = '\n'.join(f"| {k[:-17]} | {v['max_abs_diff']:.2e} | {v['box_coords_max_abs_diff_px']:.2e} | {v['class_scores_max_abs_diff']:.2e} | {'PASS' if v['pass'] else 'FAIL'} |" for k, v in avail.items())
(OUT / 'README.md').write_text(f"""# Detector ONNX export: {MODEL_NAME} ({TODAY}, v{VERSION})

Weights: `{rel(W)}` (final, per P09_Model_Freeze.json). ONNX opset {IO['opset']}, input {IO['inputs']}, output {IO['outputs']}, no NMS in the graph.

## Parity (PyTorch fp32 vs ONNX Runtime, {len(INPUTS)} inputs: {len(FRAME_PATHS)} frames + {N_RANDOM} random)
| provider | max abs diff | boxes (px) | class scores | <= {PARITY_ATOL} |
|---|---|---|---|---|
{rows}

**Acceptance (max abs diff <= {PARITY_ATOL}): {'MET' if ACCEPT['met'] else 'NOT MET'}** (max {MAXD:.2e}).

Detections after NMS, same 640x640 tensor into both, same NMS (conf {CONF}, IoU {NMS_IOU}): {SAME_T}

Detections after NMS via predict() on the image file (each backend letterboxes on its own, so inputs differ): {E2E_SUM}

Latency (ms, batch 1): {LAT}
""")
print(open(OUT / 'README.md').read())
print('files:', sorted(p.name for p in OUT.iterdir()))

# Detector ONNX export: YOLO11n_JointStrongerAug (2026-11-16, v1)

Weights: `03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt` (final, per P09_Model_Freeze.json). ONNX opset 17, input [('images', [1, 3, 640, 640])], output [('output0', [1, 12, 8400])], no NMS in the graph.

## Parity (PyTorch fp32 vs ONNX Runtime, 46 inputs: 30 frames + 16 random)
| provider | max abs diff | boxes (px) | class scores | <= 0.001 |
|---|---|---|---|---|
| CPU | 4.07e-03 | 4.07e-03 | 3.52e-06 | FAIL |
| CUDA | 3.75e-03 | 3.75e-03 | 4.02e-06 | FAIL |

**Acceptance (max abs diff <= 0.001): NOT MET** (max 4.07e-03).

Detections after NMS, same 640x640 tensor into both, same NMS (conf 0.25, IoU 0.7): {'CUDAExecutionProvider': {'pytorch_boxes': 977, 'onnx_boxes': 977, 'matched_iou_0_9': 977, 'same_class': 977, 'frames': 30, 'max_conf_diff': 4.023313522338867e-06, 'max_box_diff_px': 0.0001220703125, 'conf': 0.25, 'nms_iou': 0.7}, 'CPUExecutionProvider': {'pytorch_boxes': 977, 'onnx